In [67]:
import torch
import torch.nn as nn
import torch.optim as optim
import time

from torchvision import datasets, transforms
from torch.utils.data import Subset, DataLoader

print("PyTorch version:", torch.__version__)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Using device:", device)

PyTorch version: 2.13.0+cpu
Using device: cpu


In [69]:
class ResidualBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()

        self.conv1 = nn.Conv2d(
            channels,
            channels,
            kernel_size=3,
            padding=1,
            bias=False
        )

        self.bn1 = nn.BatchNorm2d(channels)

        self.relu = nn.ReLU(inplace=True)

        self.conv2 = nn.Conv2d(
            channels,
            channels,
            kernel_size=3,
            padding=1,
            bias=False
        )

        self.bn2 = nn.BatchNorm2d(channels)

    def forward(self, x):
        identity = x

        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)

        out = self.conv2(out)
        out = self.bn2(out)

        out = out + identity
        out = self.relu(out)

        return out

In [71]:
class ResNetCIFAR(nn.Module):
    def __init__(self, num_blocks=10, channels=128, num_classes=10):
        super().__init__()

        self.channels = channels

        self.stem = nn.Sequential(
            nn.Conv2d(
                3,
                channels,
                kernel_size=3,
                padding=1,
                bias=False
            ),
            nn.BatchNorm2d(channels),
            nn.ReLU(inplace=True)
        )

        self.blocks = nn.ModuleList([
            ResidualBlock(channels)
            for _ in range(num_blocks)
        ])

        self.global_average_pool = nn.AdaptiveAvgPool2d((1, 1))

        self.classifier = nn.Linear(
            channels,
            num_classes
        )

    def forward(self, x):
        x = self.stem(x)

        for block in self.blocks:
            x = block(x)

        x = self.global_average_pool(x)
        x = torch.flatten(x, 1)
        x = self.classifier(x)

        return x

    def get_block_representations(self, x):
        x = self.stem(x)

        representations = []

        for block in self.blocks:
            x = block(x)
            representations.append(x)

        return representations

In [75]:
model = ResNetCIFAR(
    num_blocks=10,
    channels=128,
    num_classes=10
)

model = model.to(device)

print("Model created successfully.")

Model created successfully.


In [77]:
transform = transforms.Compose([
    transforms.ToTensor()
])

train_dataset = datasets.CIFAR10(
    root="./data",
    train=True,
    download=False,
    transform=transform
)

test_dataset = datasets.CIFAR10(
    root="./data",
    train=False,
    download=False,
    transform=transform
)

print("Full training dataset:", len(train_dataset))
print("Full test dataset:", len(test_dataset))

Full training dataset: 50000
Full test dataset: 10000


In [79]:
TRAIN_SAMPLES_PER_CLASS = 500
TEST_SAMPLES_PER_CLASS = 100

train_indices = []
test_indices = []

for class_id in range(10):

    class_indices = [
        i for i, label in enumerate(train_dataset.targets)
        if label == class_id
    ]

    train_indices.extend(
        class_indices[:TRAIN_SAMPLES_PER_CLASS]
    )

for class_id in range(10):

    class_indices = [
        i for i, label in enumerate(test_dataset.targets)
        if label == class_id
    ]

    test_indices.extend(
        class_indices[:TEST_SAMPLES_PER_CLASS]
    )

small_train_dataset = Subset(
    train_dataset,
    train_indices
)

small_test_dataset = Subset(
    test_dataset,
    test_indices
)

print("Reduced training dataset:", len(small_train_dataset))
print("Reduced test dataset:", len(small_test_dataset))

Reduced training dataset: 5000
Reduced test dataset: 1000


In [81]:
BATCH_SIZE = 128

small_train_loader = DataLoader(
    small_train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0
)

small_test_loader = DataLoader(
    small_test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0
)

print("Training batches:", len(small_train_loader))
print("Testing batches:", len(small_test_loader))

Training batches: 40
Testing batches: 8


In [83]:
images, labels = next(iter(small_train_loader))

images = images.to(device)
labels = labels.to(device)

outputs = model(images)

print("Input shape:", images.shape)
print("Output shape:", outputs.shape)

Input shape: torch.Size([128, 3, 32, 32])
Output shape: torch.Size([128, 10])


In [85]:
criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=0.001
)

print("Loss:", criterion.__class__.__name__)
print("Optimizer:", optimizer.__class__.__name__)

Loss: CrossEntropyLoss
Optimizer: Adam


In [87]:
def train_one_epoch(model, loader, criterion, optimizer, device):

    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in loader:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

        running_loss += loss.item() * images.size(0)

        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)

        correct += (
            predicted == labels
        ).sum().item()

    epoch_loss = running_loss / total

    epoch_accuracy = (
        100.0 * correct / total
    )

    return epoch_loss, epoch_accuracy

In [89]:
def evaluate(model, loader, criterion, device):

    model.eval()

    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():

        for images, labels in loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            loss = criterion(outputs, labels)

            running_loss += (
                loss.item() * images.size(0)
            )

            _, predicted = torch.max(
                outputs, 1
            )

            total += labels.size(0)

            correct += (
                predicted == labels
            ).sum().item()

    loss = running_loss / total

    accuracy = (
        100.0 * correct / total
    )

    return loss, accuracy

In [91]:
start_time = time.time()

train_loss, train_accuracy = train_one_epoch(
    model,
    small_train_loader,
    criterion,
    optimizer,
    device
)

elapsed_time = time.time() - start_time

print(f"Training loss: {train_loss:.4f}")
print(f"Training accuracy: {train_accuracy:.2f}%")
print(f"Training time: {elapsed_time / 60:.2f} minutes")

Training loss: 2.1136
Training accuracy: 25.36%
Training time: 21.53 minutes


In [92]:
test_loss, test_accuracy = evaluate(
    model,
    small_test_loader,
    criterion,
    device
)

print(f"Test loss: {test_loss:.4f}")
print(f"Test accuracy: {test_accuracy:.2f}%")

Test loss: 2.1034
Test accuracy: 23.50%


In [93]:
epoch_result = {
    "epoch": 1,
    "train_loss": train_loss,
    "train_accuracy": train_accuracy,
    "test_loss": test_loss,
    "test_accuracy": test_accuracy,
    "training_time_minutes": elapsed_time / 60
}

print(epoch_result)

{'epoch': 1, 'train_loss': 2.1135765348434448, 'train_accuracy': 25.36, 'test_loss': 2.103426015853882, 'test_accuracy': 23.5, 'training_time_minutes': 21.528974596659342}


In [97]:
torch.save(
    model.state_dict(),
    "./resnet_cifar10_epoch1.pth"
)

print("Model checkpoint saved successfully.")

Model checkpoint saved successfully.
